<a href="https://colab.research.google.com/github/SantiagoAmaya99/Crack-Detector-Using-Deep-Learning-Models/blob/main/Detecci%C3%B3n_de_grietas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Para implementar este proyecto en Google Colab utilizando aprendizaje débilmente supervisado, utilizaremos **TensorFlow/Keras** y el algoritmo **Grad-CAM** (Gradient-weighted Class Activation Mapping). Este enfoque entrenará una red para clasificar si hay una grieta o no, y luego usará los gradientes de la última capa convolucional para generar un "mapa de calor" que localice la grieta sin haber usado coordenadas en el entrenamiento.

A continuación, tienes la estructura del código dividida en las celdas que debes ejecutar en tu notebook.

**Celda 1: Configuración del entorno y montaje de Google Drive**

In [3]:
from google.colab import drive
drive.mount('/content/drive')
%cd '/content/drive/MyDrive/IA2/Dataset grietas'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/IA2/Dataset grietas


**Celda 2: Carga y preprocesamiento del Dataset**
Asumiendo que tienes el dataset organizado en carpetas `entrenamiento/grieta` y `entrenamiento/sin_grieta`.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import os

# Ajusta esta ruta a la estructura exacta de carpetas que Kaggle haya descomprimido
ruta_dataset = '/content/dataset'
tamano_lote = 32
tamano_imagen = (256, 256)

# Cargar las imágenes directamente desde el disco virtual de Colab
dataset_entrenamiento = tf.keras.utils.image_dataset_from_directory(
    ruta_dataset,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=tamano_imagen,
    batch_size=tamano_lote
)

dataset_validacion = tf.keras.utils.image_dataset_from_directory(
    ruta_dataset,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=tamano_imagen,
    batch_size=tamano_lote
)

# Normalizar los valores de los píxeles
capa_normalizacion = layers.Rescaling(1./255)
dataset_entrenamiento = dataset_entrenamiento.map(lambda x, y: (capa_normalizacion(x), y))
dataset_validacion = dataset_validacion.map(lambda x, y: (capa_normalizacion(x), y))

**Celda 3: Definición del Modelo (Arquitectura MobileNetV2 adaptada)**
Usamos MobileNetV2 por ser ligera (ideal para procesamiento rápido). Es crucial incluir `GlobalAveragePooling2D` para que el modelo concentre la información espacial y permita aplicar Grad-CAM.

In [ ]:
def construir_modelo():
    # Cargar MobileNetV2 pre-entrenada sin las capas finales
    modelo_base = tf.keras.applications.MobileNetV2(
        input_shape=(256, 256, 3),
        include_top=False,
        weights='imagenet'
    )

    # Congelar el modelo base (opcional, acelera el primer entrenamiento)
    modelo_base.trainable = False

    # Añadir la cabeza de clasificación débilmente supervisada
    entradas = tf.keras.Input(shape=(256, 256, 3))
    x = modelo_base(entradas, training=False)
    x = layers.GlobalAveragePooling2D()(x) # CRÍTICO PARA LOCALIZACIÓN
    x = layers.Dropout(0.2)(x)
    salidas = layers.Dense(1, activation='sigmoid')(x) # Clasificación binaria (Grieta / No Grieta)

    modelo = tf.keras.Model(entradas, salidas)

    modelo.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return modelo, modelo_base

modelo_grietas, modelo_base = construir_modelo()
modelo_grietas.summary()

**Celda 4: Entrenamiento del clasificador**

In [ ]:
epocas = 10
historial = modelo_grietas.fit(
    dataset_entrenamiento,
    validation_data=dataset_validacion,
    epochs=epocas
)

**Celda 5: Implementación de Grad-CAM (El núcleo del proyecto)**
Esta función calcula qué partes de la imagen hicieron que la red neuronal "decidiera" que había una grieta.

In [ ]:
def generar_grad_cam(img_array, modelo, nombre_capa_conv):
    # Crear un modelo que devuelve las activaciones de la última capa convolucional y las predicciones finales
    modelo_gradcam = tf.keras.models.Model(
        [modelo.inputs],
        [modelo.get_layer(nombre_capa_conv).output, modelo.output]
    )

    # Registrar las operaciones con GradientTape
    with tf.GradientTape() as tape:
        salida_conv, predicciones = modelo_gradcam(img_array)
        clase_indice = tf.argmax(predicciones[0])
        salida_clase = predicciones[:, clase_indice]

    # Calcular los gradientes de la clase predicha con respecto a la salida convolucional
    grads = tape.gradient(salida_clase, salida_conv)

    # Promediar los gradientes espacialmente
    pesos_canales = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Multiplicar las salidas de la capa convolucional por los pesos
    salida_conv = salida_conv[0]
    mapa_calor = tf.reduce_sum(tf.multiply(salida_conv, pesos_canales), axis=-1)

    # Aplicar ReLU al mapa de calor (solo nos interesan las influencias positivas)
    mapa_calor = np.maximum(mapa_calor, 0)
    mapa_calor /= np.max(mapa_calor) # Normalizar

    return mapa_calor

**Celda 6: Visualización y Localización (El resultado final)**

In [ ]:
# Cargar una imagen de prueba
ruta_img_prueba = '/content/drive/MyDrive/imagen_prueba_grieta.jpg'
img = tf.keras.preprocessing.image.load_img(ruta_img_prueba, target_size=(256, 256))
img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array_norm = np.expand_dims(img_array, axis=0) / 255.0

# El nombre de la última capa convolucional varía según el modelo base.
# En MobileNetV2 envuelta como hicimos, la capa interna es el modelo_base.
nombre_ultima_capa = modelo_base.layers[-1].name

# Para acceder a la capa interna, usamos el modelo base dentro de nuestro modelo
mapa_calor = generar_grad_cam(img_array_norm, modelo_base, nombre_ultima_capa)

# Redimensionar el mapa de calor al tamaño de la imagen original
mapa_calor_redimensionado = cv2.resize(mapa_calor, (img_array.shape[1], img_array.shape[0]))
mapa_calor_redimensionado = np.uint8(255 * mapa_calor_redimensionado)

# Aplicar colormap
mapa_color = cv2.applyColorMap(mapa_calor_redimensionado, cv2.COLORMAP_JET)

# Superponer el mapa de calor sobre la imagen original
imagen_superpuesta = cv2.addWeighted(np.uint8(img_array), 0.6, mapa_color, 0.4, 0)

# Para su marcado manual (Ground Truth): pueden binarizar el mapa de calor
# (por ejemplo, tomar pixeles > 70% de intensidad) y extraer el Bounding Box con cv2.boundingRect()

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(np.uint8(img_array))
plt.title("Imagen Original")

plt.subplot(1, 2, 2)
plt.imshow(imagen_superpuesta)
plt.title("Localización Débilmente Supervisada (Grad-CAM)")
plt.show()